# routes.handlers

> Response builder functions for virtual collection navigation (Tier 1 API).

In [ ]:
#| default_exp routes.handlers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import inspect
from typing import Any, Callable, List, Optional, Tuple

from fasthtml.common import Hidden

from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.windowing import (
    navigate, navigate_cursor, clamp_window_start, compute_window,
    find_nearest_focusable,
)
from cjm_fasthtml_virtual_collection.components.table import render_slot, render_table_rows
from cjm_fasthtml_virtual_collection.components.footer import render_footer
from cjm_fasthtml_virtual_collection.components.scrollbar import render_scrollbar

## Response Builders

In [ ]:
#| export
def _render_window_start_oob(
    state: VirtualCollectionState,     # Current state
    ids: VirtualCollectionHtmlIds,     # HTML IDs
) -> Hidden:  # Hidden input with OOB swap
    """Render OOB hidden input carrying the current window_start for JS thumb positioning."""
    return Hidden(
        value=str(state.window_start),
        id=ids.window_start_input,
        name="window_start",
        hx_swap_oob="outerHTML",
    )


def _render_scrollbar_nav_oob(
    state: VirtualCollectionState,     # Current state
    config: VirtualCollectionConfig,   # Collection config
    ids: VirtualCollectionHtmlIds,     # HTML IDs
) -> 'Any':  # Scrollbar element with OOB swap (or None if no scrollbar configured)
    """Render OOB scrollbar. Hidden via CSS class when items fit in viewport."""
    if not config.show_scrollbar:
        return None
    sb = render_scrollbar(state, config, ids)
    if state.total_items <= state.visible_rows:
        sb.attrs['class'] = sb.attrs.get('class', '') + ' hidden'
    sb.attrs["hx-swap-oob"] = "outerHTML"
    return sb


def build_nav_response(
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (already mutated)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements (slot OOBs + footer + window_start input + scrollbar)
    """Build OOB response for navigation: all visible slots + footer + window_start + scrollbar."""
    render_start, render_end = compute_window(
        state.window_start, state.visible_rows, state.total_items
    )

    slot_oobs = [
        render_slot(
            slot_index=slot_idx, item=items[item_idx], item_index=item_idx,
            config=config, state=state, ids=ids, render_cell=render_cell,
            oob=True, focus_url=focus_url,
        )
        for slot_idx, item_idx in enumerate(range(render_start, render_end))
    ]

    result = tuple(slot_oobs) + (
        render_footer(state, ids, oob=True),
        _render_window_start_oob(state, ids),
    )

    sb_oob = _render_scrollbar_nav_oob(state, config, ids)
    if sb_oob is not None:
        result = result + (sb_oob,)

    return result

In [ ]:
#| export
def build_cursor_move_response(
    old_cursor: int,                        # Previous cursor index
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (cursor already updated)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements (affected slot OOBs + footer + window_start input + scrollbar)
    """Build OOB response for cursor-only move: swap just the affected slots."""
    oob_parts = []

    # Re-render old cursor slot (remove highlight)
    if old_cursor != state.cursor_index:
        old_slot = old_cursor - state.window_start
        oob_parts.append(render_slot(
            slot_index=old_slot, item=items[old_cursor], item_index=old_cursor,
            config=config, state=state, ids=ids, render_cell=render_cell,
            oob=True, focus_url=focus_url,
        ))

    # Re-render new cursor slot (add highlight)
    new_slot = state.cursor_index - state.window_start
    oob_parts.append(render_slot(
        slot_index=new_slot, item=items[state.cursor_index], item_index=state.cursor_index,
        config=config, state=state, ids=ids, render_cell=render_cell,
        oob=True, focus_url=focus_url,
    ))

    oob_parts.append(render_footer(state, ids, oob=True))
    oob_parts.append(_render_window_start_oob(state, ids))

    sb_oob = _render_scrollbar_nav_oob(state, config, ids)
    if sb_oob is not None:
        oob_parts.append(sb_oob)

    return tuple(oob_parts)

## Navigation Handlers (Tier 1)

In [ ]:
#| export
def _is_cursor_visible(
    state: VirtualCollectionState,  # Current state
) -> bool:  # Whether cursor is within the visible window
    """Check if the cursor index is within the current visible window."""
    if state.cursor_index < 0:
        return False
    return state.window_start <= state.cursor_index < state.window_start + state.visible_rows


def _append_cursor_change(
    result: Tuple,                          # Base response tuple
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state
    on_cursor_change: Optional[Callable],   # Callback: (item, cursor_index, state) -> Tuple
) -> Tuple:  # Response with appended cursor change OOB elements
    """Append on_cursor_change callback results to a response tuple."""
    if on_cursor_change is None:
        return result
    cursor = state.cursor_index
    if cursor < 0 or cursor >= len(items):
        return result
    extra = on_cursor_change(items[cursor], cursor, state)
    if extra:
        return result + tuple(extra) if not isinstance(extra, tuple) else result + extra
    return result


def _scroll_to_cursor(
    state: VirtualCollectionState,  # Current state (mutated in place)
) -> None:
    """Scroll window to bring an off-screen cursor into view."""
    if state.cursor_index < 0:
        return
    window_end = state.window_start + state.visible_rows
    if state.cursor_index >= window_end:
        state.window_start = state.cursor_index - state.visible_rows + 1
    elif state.cursor_index < state.window_start:
        state.window_start = state.cursor_index
    state.window_start = clamp_window_start(
        state.window_start, state.visible_rows, state.total_items
    )


def handle_navigate(
    direction: str,                         # 'up', 'down', 'page_up', 'page_down', 'first', 'last'
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
    is_skippable: Optional[Callable[[Any], bool]] = None,  # Predicate: item -> skip?
    on_cursor_change: Optional[Callable] = None,  # Callback: (item, cursor_index, state) -> Tuple
) -> Tuple:  # OOB elements
    """Navigate in a direction. Mutates state in place."""
    if state.total_items == 0:
        return ()

    # Build index-based skip predicate from item-based one
    _skip_idx = (lambda idx: is_skippable(items[idx])) if is_skippable else None

    old_cursor = state.cursor_index

    if direction in ('up', 'down'):
        # If cursor is off-screen, scroll window to bring it into view
        # then proceed with normal navigation from the now-visible cursor
        if not _is_cursor_visible(state):
            _scroll_to_cursor(state)
            result = build_nav_response(items, state, config, ids, render_cell, focus_url)
            return result

        new_cursor, new_window_start, window_changed = navigate_cursor(
            state.cursor_index, direction, state.window_start,
            state.visible_rows, state.total_items, is_skippable=_skip_idx,
        )
        state.cursor_index = new_cursor
        state.window_start = new_window_start

        if window_changed:
            result = build_nav_response(items, state, config, ids, render_cell, focus_url)
        else:
            result = build_cursor_move_response(old_cursor, items, state, config, ids, render_cell, focus_url)
    else:
        # Page/Home/End navigation
        state.window_start = navigate(
            state.window_start, direction, state.visible_rows, state.total_items
        )
        # Move cursor to nearest focusable in the new window
        if direction in ('first', 'page_up'):
            target = state.window_start
            search_dir = 1
        else:
            target = min(state.window_start + state.visible_rows - 1, state.total_items - 1)
            search_dir = -1
        state.cursor_index = find_nearest_focusable(
            target, state.total_items, _skip_idx, direction=search_dir,
        )
        result = build_nav_response(items, state, config, ids, render_cell, focus_url)

    if state.cursor_index != old_cursor:
        result = _append_cursor_change(result, items, state, on_cursor_change)
    return result

In [ ]:
#| export
def _call_action_callback(
    callback: Callable,  # Consumer callback to invoke
    item: Any,  # Item at the cursor position
    row_index: int,  # Row index
    state: VirtualCollectionState,  # Current VC state
    request: Any = None,  # FastHTML request (passed if callback accepts it)
) -> Any:  # Callback result
    """Call an action callback, passing request if the callback signature accepts it."""
    sig = inspect.signature(callback)
    if 'request' in sig.parameters and request is not None:
        return callback(item, row_index, state, request=request)
    return callback(item, row_index, state)

In [ ]:
#| export
def handle_navigate_to_index(
    target_index: int,                      # Target window_start
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements
    """Navigate to a specific index. Mutates state.window_start in place."""
    if state.total_items == 0:
        return ()
    state.window_start = clamp_window_start(
        target_index, state.visible_rows, state.total_items
    )
    return build_nav_response(items, state, config, ids, render_cell, focus_url)

In [ ]:
#| export
def _render_scrollbar_oob(
    state: VirtualCollectionState,     # Current state
    config: VirtualCollectionConfig,   # Collection config
    ids: VirtualCollectionHtmlIds,     # HTML IDs
) -> Any:  # Scrollbar element with OOB swap (or None if no scrollbar configured)
    """Render OOB scrollbar. Hidden via CSS class when items fit in viewport."""
    if not config.show_scrollbar:
        return None
    sb = render_scrollbar(state, config, ids)
    if state.total_items <= state.visible_rows:
        sb.attrs['class'] = sb.attrs.get('class', '') + ' hidden'
    sb.attrs["hx-swap-oob"] = "outerHTML"
    return sb


def _build_container_response(
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements (container + footer + window_start input [+ scrollbar])
    """Build OOB response that replaces the entire rows container with new slots."""
    rows_oob = render_table_rows(items, config, state, ids, render_cell, focus_url=focus_url)
    rows_oob.attrs["hx-swap-oob"] = "outerHTML"
    result = (
        rows_oob,
        render_footer(state, ids, oob=True),
        _render_window_start_oob(state, ids),
    )
    sb_oob = _render_scrollbar_oob(state, config, ids)
    if sb_oob is not None:
        result = result + (sb_oob,)
    return result


def handle_update_viewport(
    visible_rows: int,                      # New visible row count
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    is_auto: bool = True,                   # Whether from auto-fit
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements
    """Update viewport with new row count. Mutates state in place."""
    visible_rows = max(1, visible_rows)
    state.visible_rows = visible_rows
    state.is_auto_mode = is_auto
    state.window_start = clamp_window_start(
        state.window_start, state.visible_rows, state.total_items
    )
    # Keep cursor visible after resize
    _scroll_to_cursor(state)
    # Replace entire container — slot count changed
    return _build_container_response(items, state, config, ids, render_cell, focus_url)

In [ ]:
#| export
def handle_focus_row(
    row_index: int,                         # Row index to focus
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
    on_refocus: Optional[Callable] = None,  # Callback when clicking already-focused row: (item, row_index, state) -> Tuple
    is_skippable: Optional[Callable[[Any], bool]] = None,  # Predicate: item -> skip?
    on_cursor_change: Optional[Callable] = None,  # Callback: (item, cursor_index, state) -> Tuple
    request: Any = None,  # FastHTML request (passed to on_refocus if it accepts it)
) -> Tuple:  # OOB elements (affected slot OOBs + footer + window_start input)
    """Move cursor to a specific row via click/tap.

    If the clicked row is skippable, returns empty (no-op).
    If `on_refocus` is provided and the clicked row is already the cursor,
    delegates to `on_refocus` instead of the normal cursor-move logic.
    """
    if state.total_items == 0:
        return ()
    clamped = max(0, min(row_index, state.total_items - 1))

    # Reject focus on skippable items
    if is_skippable is not None and is_skippable(items[clamped]):
        return ()

    # Refocus: clicked row is already the cursor
    if on_refocus is not None and clamped == state.cursor_index:
        return _call_action_callback(on_refocus, items[clamped], clamped, state, request)

    old_cursor = state.cursor_index
    state.cursor_index = clamped
    result = build_cursor_move_response(old_cursor, items, state, config, ids, render_cell, focus_url)
    if state.cursor_index != old_cursor:
        result = _append_cursor_change(result, items, state, on_cursor_change)
    return result

In [ ]:
#| export
def handle_activate(
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    on_activate: Callable,                  # Consumer callback: (item, row_index, state[, request]) -> Tuple of OOB elements
    focus_url: str = "",                    # URL for click-to-focus
    request: Any = None,  # FastHTML request (passed to on_activate if it accepts it)
) -> Tuple:  # OOB elements from consumer callback
    """Activate the focused row via Space/Enter. Delegates to consumer callback."""
    cursor = state.cursor_index
    if cursor < 0 or cursor >= state.total_items:
        return ()
    if not _is_cursor_visible(state):
        return ()
    return _call_action_callback(on_activate, items[cursor], cursor, state, request)

In [ ]:
#| export
def handle_sort(
    column_key: str,                        # Column key to sort by
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    sort_callback: Callable,                # Consumer: (items, column_key, ascending) -> sorted items
    sort_url: str = "",                     # Sort URL for header re-render
    focus_url: str = "",                    # URL for click-to-focus
    is_skippable: Optional[Callable[[Any], bool]] = None,  # Predicate: item -> skip?
    on_cursor_change: Optional[Callable] = None,  # Callback: (item, cursor_index, state) -> Tuple
) -> Tuple:  # OOB elements (header + rows + footer + window_start)
    """Sort by column. Toggles direction if same column, resets window to start."""
    old_cursor = state.cursor_index

    if state.sort_column == column_key:
        state.sort_ascending = not state.sort_ascending
    else:
        state.sort_column = column_key
        state.sort_ascending = True

    # Consumer sorts items in place
    sort_callback(items, column_key, state.sort_ascending)

    # Reset to top and find nearest focusable cursor position
    state.window_start = 0
    _skip_idx = (lambda idx: is_skippable(items[idx])) if is_skippable else None
    state.cursor_index = find_nearest_focusable(0, state.total_items, _skip_idx, direction=1)

    # Re-render header with updated sort indicators
    from cjm_fasthtml_virtual_collection.components.table import render_header_row
    header_oob = render_header_row(config, ids, state=state, sort_url=sort_url)
    header_oob.attrs["hx-swap-oob"] = "outerHTML"

    # Re-render visible rows
    nav_response = build_nav_response(items, state, config, ids, render_cell, focus_url)

    result = (header_oob,) + nav_response
    if state.cursor_index != old_cursor:
        result = _append_cursor_change(result, items, state, on_cursor_change)
    return result

## External Item Mutation

Public handler for when the consumer modifies the item list externally (delete, add, filter). Synchronizes VC state with the new item count and rebuilds the rows container.

In [ ]:
#| export
def build_items_changed_response(
    items:list,  # Full item list (already mutated by consumer)
    state:VirtualCollectionState,  # Current state (mutated in place)
    config:VirtualCollectionConfig,  # Collection config
    ids:VirtualCollectionHtmlIds,  # HTML IDs
    render_cell:Callable,  # Consumer cell render callback
    focus_url:str="",  # URL for click-to-focus
    is_skippable:Optional[Callable[[Any], bool]]=None,  # Predicate: item -> skip?
    refit_callback:str="",  # JS auto-fit call expression (from auto_fit_callback_name)
) -> Tuple:  # OOB elements (container + scrollbar + footer + window_start [+ refit trigger])
    """Rebuild the VC after the consumer modifies the item list externally.

    Updates total_items, clamps window_start and cursor_index, then replaces
    the entire rows container via OOB. Use this after deleting, adding, or
    filtering items — any operation that changes the item count.

    Pass `refit_callback` (from `auto_fit_callback_name(config)`) to trigger
    auto-fit re-evaluation after the update — needed when items are added and
    the viewport may have room for more rows than currently rendered.
    """
    # Capture pre-mutation scrollbar visibility for refit flash prevention
    _was_scrollbar_hidden = state.total_items <= state.visible_rows
    
    state.total_items = len(items)
    
    if state.total_items == 0:
        state.window_start = 0
        state.cursor_index = -1
    else:
        state.window_start = clamp_window_start(
            state.window_start, state.visible_rows, state.total_items,
        )
        # Clamp cursor to valid range, then find nearest focusable
        _skip_idx = (lambda idx: is_skippable(items[idx])) if is_skippable else None
        if state.cursor_index >= state.total_items:
            state.cursor_index = state.total_items - 1
        if state.cursor_index < 0:
            state.cursor_index = 0
        state.cursor_index = find_nearest_focusable(
            state.cursor_index, state.total_items, _skip_idx, direction=-1,
        )
    
    result = _build_container_response(items, state, config, ids, render_cell, focus_url)
    
    if refit_callback:
        from fasthtml.common import Div, Script
        
        # Preserve pre-mutation scrollbar visibility during refit to prevent
        # flash. Auto-fit will set the correct final state after it settles.
        # - Was hidden → keep hidden (prevents flash-on during growth cycle)
        # - Was visible → keep visible (prevents flash-off during add/delete)
        if config.show_scrollbar and refit_callback:
            result_list = list(result)
            for i, el in enumerate(result_list):
                if hasattr(el, 'attrs') and el.attrs.get('id') == ids.scrollbar_track:
                    cls = el.attrs.get('class', '')
                    is_hidden = ' hidden' in cls or cls.endswith('hidden')
                    if _was_scrollbar_hidden and not is_hidden:
                        el.attrs['class'] = cls + ' hidden'
                    elif not _was_scrollbar_hidden and is_hidden:
                        el.attrs['class'] = cls.replace(' hidden', '')
                    break
            result = tuple(result_list)
        
        # OOB-swap the refit trigger div with a script that calls auto-fit.
        # HTMX executes scripts in OOB-swapped content.
        trigger = Div(
            Script(f"setTimeout(() => {{ {refit_callback}; }}, 0);"),
            id=ids.refit_trigger,
            style="display:none",
        )
        trigger.attrs['hx-swap-oob'] = 'innerHTML'
        result = result + (trigger,)
    
    return result

## Tests

In [ ]:
from dataclasses import dataclass
from fasthtml.common import to_xml, Span
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, ColumnDef,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

@dataclass
class Item:
    name: str

test_items = [Item(f"file_{i}.txt") for i in range(50)]
columns = (ColumnDef(key="name", header="Name"),)
config = VirtualCollectionConfig(prefix="th", columns=columns)
ids = VirtualCollectionHtmlIds(prefix="th")

def test_render_cell(item, ctx):
    return Span(item.name)

In [ ]:
# Test handle_navigate down — cursor moves within window (no scroll)
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0, cursor_index=0)
result = handle_navigate("down", test_items, state, config, ids, test_render_cell)
assert state.cursor_index == 1
assert state.window_start == 0

# Response: 2 slot OOBs + footer + window_start + scrollbar = 5
assert len(result) == 5
old_slot_html = to_xml(result[0])
assert 'id="th-slot-0"' in old_slot_html
assert 'hx-swap-oob="innerHTML"' in old_slot_html
new_slot_html = to_xml(result[1])
assert 'id="th-slot-1"' in new_slot_html
assert 'hx-swap-oob="innerHTML"' in new_slot_html
# Inner row has cursor highlight
assert 'bg-base-300' in new_slot_html
assert 'bg-base-300' not in old_slot_html
print("handle_navigate down (cursor move) test passed")

In [ ]:
# Test handle_navigate down — cursor at bottom edge triggers scroll
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0, cursor_index=4)
result = handle_navigate("down", test_items, state, config, ids, test_render_cell)
assert state.cursor_index == 5
assert state.window_start == 1

# Response: 5 slot OOBs + footer + window_start + scrollbar = 8
assert len(result) == 8
# All slots have innerHTML OOB
for i in range(5):
    slot_html = to_xml(result[i])
    assert f'id="th-slot-{i}"' in slot_html
    assert 'hx-swap-oob="innerHTML"' in slot_html
# Cursor is at bottom slot (slot 4 = item 5)
assert 'bg-base-300' in to_xml(result[4])
assert 'bg-base-300' not in to_xml(result[0])
print("handle_navigate down (scroll at edge) test passed")

# Test handle_navigate up — clamped at top
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0, cursor_index=0)
result = handle_navigate("up", test_items, state, config, ids, test_render_cell)
assert state.cursor_index == 0
assert state.window_start == 0
print("handle_navigate up (clamped at top) test passed")

In [ ]:
# Test handle_navigate_to_index
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0)
result = handle_navigate_to_index(25, test_items, state, config, ids, test_render_cell)
assert state.window_start == 25

rows_html = to_xml(result[0])
assert 'file_25.txt' in rows_html
print("handle_navigate_to_index test passed")

handle_navigate_to_index test passed


In [ ]:
# Test handle_update_viewport — replaces entire container + scrollbar OOB
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=46)
result = handle_update_viewport(10, test_items, state, config, ids, test_render_cell)
assert state.visible_rows == 10
assert state.window_start == 40  # re-clamped: 50 - 10 = 40

# Response: container + footer + window_start + scrollbar = 4 parts
assert len(result) == 4
container_html = to_xml(result[0])
assert 'hx-swap-oob="outerHTML"' in container_html
assert f'id="{ids.rows}"' in container_html
# Contains 10 slots
assert 'id="th-slot-0"' in container_html
assert 'id="th-slot-9"' in container_html
assert 'id="th-slot-10"' not in container_html

# Scrollbar OOB has fresh data attributes (visible: 50 > 10)
scrollbar_html = to_xml(result[3])
assert f'id="{ids.scrollbar_track}"' in scrollbar_html
assert 'hx-swap-oob="outerHTML"' in scrollbar_html
assert 'hidden' not in scrollbar_html

print("handle_update_viewport test passed")

# Test handle_update_viewport when total_items <= visible_rows — scrollbar hidden
state_small = VirtualCollectionState(total_items=3, visible_rows=1, window_start=0, cursor_index=0)
small_items = test_items[:3]
result_small = handle_update_viewport(10, small_items, state_small, config, ids, test_render_cell)
assert state_small.visible_rows == 10
# Response still 4 parts — scrollbar present but hidden
assert len(result_small) == 4
scrollbar_small = to_xml(result_small[3])
assert 'hidden' in scrollbar_small
print("handle_update_viewport (scrollbar hidden) test passed")

In [ ]:
# Test edge case: empty collection — all handlers return empty tuple
state_empty = VirtualCollectionState(total_items=0, visible_rows=5, cursor_index=0)
assert handle_navigate("down", [], state_empty, config, ids, test_render_cell) == ()
assert handle_navigate("up", [], state_empty, config, ids, test_render_cell) == ()
assert handle_navigate("page_down", [], state_empty, config, ids, test_render_cell) == ()
assert handle_navigate_to_index(0, [], state_empty, config, ids, test_render_cell) == ()
assert handle_focus_row(0, [], state_empty, config, ids, test_render_cell) == ()
print("Empty collection edge case test passed")

In [ ]:
# Test handle_focus_row with on_refocus — clicking already-focused row
from fasthtml.common import Div

refocus_called = []
def test_on_refocus(item, row_index, st):
    refocus_called.append((item.name, row_index))
    return (Div("refocused", id="refocus-result"),)

state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0, cursor_index=3)
result = handle_focus_row(
    3, test_items, state, config, ids, test_render_cell,
    on_refocus=test_on_refocus,
)
assert len(refocus_called) == 1
assert refocus_called[0] == ("file_3.txt", 3)
assert 'refocused' in to_xml(result[0])
print("handle_focus_row on_refocus test passed")

# Clicking a different row still does normal cursor move (no refocus)
refocus_called.clear()
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0, cursor_index=2)
result = handle_focus_row(
    4, test_items, state, config, ids, test_render_cell,
    on_refocus=test_on_refocus,
)
assert len(refocus_called) == 0
assert state.cursor_index == 4
print("handle_focus_row normal focus (not refocus) test passed")

In [ ]:
# Test build_items_changed_response — delete items, state clamps correctly
state = VirtualCollectionState(total_items=50, visible_rows=10, window_start=40, cursor_index=45)
# Simulate deleting 30 items (now only 20 remain)
remaining = test_items[:20]
result = build_items_changed_response(remaining, state, config, ids, test_render_cell)

assert state.total_items == 20
assert state.window_start == 10  # clamped: 20 - 10 = 10
assert state.cursor_index == 19  # clamped from 45 to 19 (last item)
# container + footer + window_start + scrollbar = 4 (scrollbar visible: 20 > 10)
assert len(result) == 4
container_html = to_xml(result[0])
assert 'hx-swap-oob="outerHTML"' in container_html
assert f'id="{ids.rows}"' in container_html
scrollbar_html = to_xml(result[3])
assert 'hidden' not in scrollbar_html
print("build_items_changed_response (shrink) test passed")

# Test delete to empty
state2 = VirtualCollectionState(total_items=5, visible_rows=10, window_start=0, cursor_index=2)
result2 = build_items_changed_response([], state2, config, ids, test_render_cell)
assert state2.total_items == 0
assert state2.cursor_index == -1
assert state2.window_start == 0
# container + footer + window_start + scrollbar(hidden) = 4
assert len(result2) == 4
assert 'hidden' in to_xml(result2[3])
print("build_items_changed_response (empty) test passed")

# Test delete within visible window — scrollbar hidden when items fit
state3 = VirtualCollectionState(total_items=10, visible_rows=10, window_start=0, cursor_index=3)
remaining3 = test_items[:9]  # removed 1 item
result3 = build_items_changed_response(remaining3, state3, config, ids, test_render_cell)
assert state3.total_items == 9
assert state3.cursor_index == 3  # still valid, unchanged
assert state3.window_start == 0
# container + footer + window_start + scrollbar(hidden) = 4 (9 <= 10)
assert len(result3) == 4
assert 'hidden' in to_xml(result3[3])
print("build_items_changed_response (minor shrink, scrollbar hidden) test passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()